# MiniDiLoCo – Distributed test

Run this notebook **only with torchrun / a multi-GPU environment**.
For a single-GPU quick test set `WORLD_SIZE=1` and use backend `'gloo'`.

```bash
torchrun --nproc_per_node=<N_GPUS> test.py
```

Alternatively execute the cells below inside a process launched by torchrun.

In [ ]:
import os
from utils import setup, cleanup
from model import get_gpt2_model
from data import get_dataloader
from train import Trainer
from strategy import Diloco, make_adamw, make_sgd

# ── environment (set automatically by torchrun) ──────────────────────────────
os.environ.setdefault('MASTER_ADDR', 'localhost')
os.environ.setdefault('MASTER_PORT', '12355')
os.environ.setdefault('LOCAL_RANK', '0')
os.environ.setdefault('WORLD_SIZE', '1')

BACKEND    = 'nccl'          # use 'gloo' if no GPU / for CPU debugging
RANK       = int(os.environ['LOCAL_RANK'])
WORLD_SIZE = int(os.environ['WORLD_SIZE'])

setup(BACKEND, RANK, WORLD_SIZE)

In [ ]:
model      = get_gpt2_model()
dataloader = get_dataloader(rank=RANK, world_size=WORLD_SIZE)

# ── pluggable optimizers ─────────────────────────────────────────────────────
strategy = Diloco(
    inner_optimizer_factory=make_adamw(lr=4e-4, weight_decay=0.1),
    outer_optimizer_factory=make_sgd(lr=0.7, momentum=0.9, nesterov=True),
    warmup_steps=1000,
    H=500,
)

trainer = Trainer(RANK, WORLD_SIZE, model, dataloader)
history = trainer.train(strategy, total_steps=20)

cleanup()
print('Training complete.')